# COM713 – Advanced Data Structures and Algorithms
## Medical Text Classification using Advanced Data Structures & NLP
**Author:** [Mohammed Azardeen]
**Date:** 2026

This notebook contains the complete implementation of the Medical Text Classification pipeline, including custom data structures (Trie, Min-Heap, BST) and a custom TF-IDF engine.

### 1. Imports & Setup

In [1]:
import math
import re
import csv
import random
import time
from collections import defaultdict, Counter, deque
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

### 2. Data Structures Implementation
In this section, we implement the core data structures: **Trie**, **Priority Queue (Min-Heap)**, and **Binary Search Tree (BST)**.

#### 2.1 Trie (Prefix Tree)

In [ ]:
"""
Trie (Prefix Tree) Data Structure for Medical Text Classification
=================================================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module implements a Trie data structure specifically designed for
efficient medical terminology lookup, prefix-based searching, and
category-frequency tracking. The Trie enables O(m) search time where
m is the length of the search term, compared to O(n) for linear search
through a vocabulary list.

Key Features:
    - O(m) insertion and exact-match search
    - Prefix-based search for autocomplete functionality
    - Category frequency tracking per word (which medical categories use this term)
    - Word counting and vocabulary size tracking

References:
    - Cormen, T.H. et al. (2009) Introduction to Algorithms, 3rd edn.
    - Knuth, D.E. (1997) The Art of Computer Programming, Vol. 3.
"""


class TrieNode:
    """
    Represents a single node in the Trie data structure.

    Each node stores:
        - children: A dictionary mapping characters to child TrieNode objects.
                    Using a dictionary provides O(1) average-case child lookup.
        - is_end_of_word: Boolean flag indicating if this node marks the
                         end of a complete word in the Trie.
        - word_count: Number of times this complete word has been inserted.
                     Useful for term frequency tracking.
        - category_freq: Dictionary tracking how many times this word appears
                        in each medical category (e.g., {'Neoplasms': 15, 'Cardiovascular': 8}).

    Time Complexity:
        - __init__: O(1)

    Space Complexity:
        - O(1) per node (excluding children references)
    """

    def __init__(self):
        """Initialise a new TrieNode with empty children and default counters."""
        # Dictionary of character -> TrieNode mappings for child nodes
        # Using dict instead of fixed-size array for memory efficiency
        # with variable character sets (medical text includes letters, digits, hyphens)
        self.children = {}

        # Flag to mark if this node represents the end of a complete word
        self.is_end_of_word = False

        # Counter for how many times this word has been inserted
        # Useful for calculating term frequency (TF) in TF-IDF
        self.word_count = 0

        # Dictionary mapping category names to frequency counts
        # Tracks which medical categories this word appears in and how often
        self.category_freq = {}


class MedicalTrie:
    """
    A Trie (prefix tree) data structure optimised for medical text classification.

    The MedicalTrie provides efficient storage and retrieval of medical terminology,
    supporting operations essential for the text classification pipeline:
        1. Fast exact-match word lookup for vocabulary checking
        2. Prefix-based searching for medical term autocomplete
        3. Category frequency tracking to understand term-category associations
        4. Vocabulary enumeration for feature extraction

    This data structure addresses a key gap identified in the reference article:
    traditional NLP pipelines use linear search (O(n)) for vocabulary lookup,
    whereas the Trie provides O(m) lookup where m is the word length, regardless
    of vocabulary size.

    Attributes:
        root (TrieNode): The root node of the Trie (represents empty prefix).
        size (int): Total number of unique words stored in the Trie.
        total_insertions (int): Total number of insert operations performed.

    Example Usage:
        >>> trie = MedicalTrie()
        >>> trie.insert("carcinoma", "Neoplasms")
        >>> trie.insert("cardiac", "Cardiovascular")
        >>> trie.search("carcinoma")
        True
        >>> trie.starts_with("car")
        ['carcinoma', 'cardiac']
    """

    def __init__(self):
        """Initialise an empty MedicalTrie with a root node."""
        # Root node represents the empty prefix ""
        self.root = TrieNode()

        # Counter for unique words in the Trie
        self.size = 0

        # Counter for total insertions (including duplicates)
        self.total_insertions = 0

    def insert(self, word, category=None):
        """
        Insert a word into the Trie with optional category tracking.

        This method traverses the Trie character by character, creating new
        nodes as needed, and marks the final node as the end of a word.
        If a category is provided, it updates the category frequency counter
        for this word.

        Algorithm:
            1. Start at root node
            2. For each character in the word:
               a. If character not in current node's children, create new TrieNode
               b. Move to the child node for this character
            3. Mark the final node as end_of_word
            4. Increment word_count
            5. Update category frequency if category provided

        Args:
            word (str): The word to insert (will be converted to lowercase).
            category (str, optional): The medical category this word belongs to.
                                     Used for category-frequency tracking.

        Time Complexity: O(m) where m is the length of the word.
        Space Complexity: O(m) in worst case (all new nodes created).

        Returns:
            None
        """
        # Validate input
        if not word or not isinstance(word, str):
            return

        # Convert to lowercase for case-insensitive matching
        word = word.lower().strip()

        if not word:
            return

        # Start traversal from the root node
        current_node = self.root

        # Traverse character by character, creating nodes as needed
        for char in word:
            # If this character doesn't have a child node, create one
            if char not in current_node.children:
                current_node.children[char] = TrieNode()

            # Move to the child node for this character
            current_node = current_node.children[char]

        # Mark the end of the word
        # Only increment size counter if this is a new word
        if not current_node.is_end_of_word:
            current_node.is_end_of_word = True
            self.size += 1

        # Increment word count (tracks duplicates for term frequency)
        current_node.word_count += 1
        self.total_insertions += 1

        # Update category frequency if a category is provided
        if category:
            if category not in current_node.category_freq:
                current_node.category_freq[category] = 0
            current_node.category_freq[category] += 1

    def search(self, word):
        """
        Search for an exact word match in the Trie.

        Traverses the Trie character by character. Returns True only if
        the complete word exists and its final node is marked as end_of_word.

        Args:
            word (str): The word to search for (case-insensitive).

        Time Complexity: O(m) where m is the length of the word.
        Space Complexity: O(1) — no additional space used.

        Returns:
            bool: True if the word exists in the Trie, False otherwise.
        """
        node = self._find_node(word)
        return node is not None and node.is_end_of_word

    def _find_node(self, prefix):
        """
        Internal helper: traverse the Trie to find the node for a given prefix.

        Args:
            prefix (str): The prefix string to traverse.

        Time Complexity: O(m) where m is the length of the prefix.
        Space Complexity: O(1).

        Returns:
            TrieNode or None: The node at the end of the prefix path,
                             or None if the prefix doesn't exist.
        """
        if not prefix or not isinstance(prefix, str):
            return None

        prefix = prefix.lower().strip()
        current_node = self.root

        for char in prefix:
            if char not in current_node.children:
                return None  # Prefix doesn't exist in Trie
            current_node = current_node.children[char]

        return current_node

    def starts_with(self, prefix):
        """
        Find all words in the Trie that start with the given prefix.

        This is particularly useful for medical term autocomplete — given
        a prefix like "cardio", it returns all matching terms such as
        "cardiovascular", "cardiomyopathy", "cardiology", etc.

        Algorithm:
            1. Navigate to the node representing the prefix
            2. Perform DFS from that node to collect all complete words

        Args:
            prefix (str): The prefix to search for (case-insensitive).

        Time Complexity: O(m + k) where m is prefix length and k is the
                        number of matching words (DFS traversal).
        Space Complexity: O(k) for storing results + O(h) for recursion stack
                         where h is the maximum word length.

        Returns:
            list[str]: A list of all words starting with the given prefix.
        """
        results = []
        node = self._find_node(prefix)

        if node is None:
            return results

        # DFS to collect all words from this node
        prefix = prefix.lower().strip()
        self._dfs_collect_words(node, prefix, results)

        return results

    def _dfs_collect_words(self, node, current_prefix, results):
        """
        Internal helper: Depth-first search to collect all complete words
        from a given node.

        Args:
            node (TrieNode): Current node in the DFS traversal.
            current_prefix (str): The prefix built up so far.
            results (list): Accumulator list for found words.

        Time Complexity: O(k) where k is the number of nodes visited.
        Space Complexity: O(h) recursion stack depth.
        """
        # If this node marks the end of a word, add it to results
        if node.is_end_of_word:
            results.append(current_prefix)

        # Recursively explore all children (sorted for deterministic output)
        for char in sorted(node.children.keys()):
            self._dfs_collect_words(
                node.children[char],
                current_prefix + char,
                results
            )

    def get_category_distribution(self, word):
        """
        Get the category frequency distribution for a specific word.

        Returns how many times the word appears in each medical category,
        which is valuable for understanding term-category associations
        and for building category-specific feature weights.

        Args:
            word (str): The word to query (case-insensitive).

        Time Complexity: O(m) where m is the word length.
        Space Complexity: O(c) where c is the number of categories.

        Returns:
            dict: A dictionary mapping category names to frequency counts.
                  Returns empty dict if word not found.

        Example:
            >>> trie.get_category_distribution("tumor")
            {'Neoplasms': 42, 'General Pathological': 5}
        """
        node = self._find_node(word)
        if node is not None and node.is_end_of_word:
            return dict(node.category_freq)  # Return a copy
        return {}

    def get_word_count(self, word):
        """
        Get the total insertion count for a specific word.

        Useful for term frequency (TF) calculation in TF-IDF.

        Args:
            word (str): The word to query.

        Time Complexity: O(m) where m is the word length.

        Returns:
            int: Number of times the word was inserted, or 0 if not found.
        """
        node = self._find_node(word)
        if node is not None and node.is_end_of_word:
            return node.word_count
        return 0

    def get_all_words(self):
        """
        Retrieve all words stored in the Trie.

        Performs a complete DFS traversal from the root to enumerate
        the entire vocabulary. Useful for building feature vectors.

        Time Complexity: O(N) where N is the total number of nodes.
        Space Complexity: O(W × L) where W is the number of words
                         and L is the average word length.

        Returns:
            list[str]: A sorted list of all words in the Trie.
        """
        results = []
        self._dfs_collect_words(self.root, "", results)
        return results

    def delete(self, word):
        """
        Delete a word from the Trie.

        Marks the word's end node as not-end-of-word and prunes
        unnecessary nodes (nodes that have no other children and
        are not end-of-word for another word).

        Args:
            word (str): The word to delete.

        Time Complexity: O(m) where m is the word length.
        Space Complexity: O(m) for the recursion stack.

        Returns:
            bool: True if the word was found and deleted, False otherwise.
        """
        if not word:
            return False

        word = word.lower().strip()
        return self._delete_recursive(self.root, word, 0)

    def _delete_recursive(self, node, word, depth):
        """
        Internal helper: recursively delete a word and prune empty branches.

        Args:
            node (TrieNode): Current node.
            word (str): The word being deleted.
            depth (int): Current depth in the word (character index).

        Returns:
            bool: True if the current node should be deleted by its parent.
        """
        # Base case: reached the end of the word
        if depth == len(word):
            if not node.is_end_of_word:
                return False  # Word doesn't exist
            node.is_end_of_word = False
            node.word_count = 0
            node.category_freq = {}
            self.size -= 1
            # Return True if node has no children (safe to delete)
            return len(node.children) == 0

        char = word[depth]
        if char not in node.children:
            return False  # Word doesn't exist

        # Recurse to the next character
        should_delete_child = self._delete_recursive(
            node.children[char], word, depth + 1
        )

        # If child should be deleted, remove it
        if should_delete_child:
            del node.children[char]
            # Return True if this node is also deletable
            return not node.is_end_of_word and len(node.children) == 0

        return False

    def __len__(self):
        """Return the number of unique words in the Trie."""
        return self.size

    def __contains__(self, word):
        """Support 'in' operator: 'word in trie'."""
        return self.search(word)

    def __repr__(self):
        """String representation of the Trie."""
        return f"MedicalTrie(size={self.size}, total_insertions={self.total_insertions})"


#### 2.2 Priority Queue (Min-Heap)

In [ ]:
"""
Priority Queue (Min-Heap) Data Structure for Medical Text Classification
========================================================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module implements a Priority Queue using a binary min-heap, designed
for efficient top-K feature selection and category ranking in the medical
text classification pipeline.

The min-heap enables O(n log k) top-K selection compared to O(n log n)
for a full sort, which is significant when selecting the most relevant
features from a large vocabulary.

Key Features:
    - O(log n) push and pop operations
    - O(1) peek at minimum element
    - O(n log k) top-K selection
    - Custom comparison via MinHeapItem wrapper

References:
    - Cormen, T.H. et al. (2009) Introduction to Algorithms, 3rd edn.
    - Williams, J.W.J. (1964) 'Algorithm 232: Heapsort', CACM.
"""


class MinHeapItem:
    """
    Wrapper class for items stored in the priority queue.

    Encapsulates a score-item pair and provides comparison operators
    for heap ordering. The heap is ordered by score (ascending),
    so the item with the lowest score is at the top.

    For top-K maximum selection, we use a min-heap of size K:
    the heap root holds the smallest score among the current top-K,
    and new items replace it only if they have a higher score.

    Attributes:
        score (float): The priority/relevance score of the item.
        item (any): The actual data being stored (e.g., term name, category).

    Time Complexity:
        - All comparison operations: O(1)
    """

    def __init__(self, score, item):
        """
        Initialise a MinHeapItem with a score and associated data.

        Args:
            score (float): The priority score (lower = higher priority in min-heap).
            item (any): The data payload associated with this score.
        """
        self.score = score
        self.item = item

    def __lt__(self, other):
        """Less-than comparison based on score (for heap ordering)."""
        return self.score < other.score

    def __le__(self, other):
        """Less-than-or-equal comparison based on score."""
        return self.score <= other.score

    def __gt__(self, other):
        """Greater-than comparison based on score."""
        return self.score > other.score

    def __ge__(self, other):
        """Greater-than-or-equal comparison based on score."""
        return self.score >= other.score

    def __eq__(self, other):
        """Equality comparison based on score."""
        if not isinstance(other, MinHeapItem):
            return False
        return self.score == other.score

    def __repr__(self):
        """String representation for debugging."""
        return f"MinHeapItem(score={self.score:.4f}, item={self.item})"


class PriorityQueue:
    """
    A priority queue implementation using a binary min-heap.

    This data structure is used in the medical text classification pipeline for:
        1. Top-K feature selection: Efficiently selecting the K most important
           TF-IDF features from a large vocabulary.
        2. Category ranking: Ranking predicted categories by confidence score
           to determine the most likely medical condition.
        3. Term relevance ranking: In the explainability module, ranking which
           terms contributed most to a classification decision.

    The min-heap maintains the heap property: parent.score <= child.score.
    This enables O(1) access to the minimum element and O(log n) insertion
    and removal.

    For top-K maximum selection, we maintain a min-heap of capacity K.
    New items are only inserted if they score higher than the current minimum,
    ensuring we always have the K highest-scoring items.

    Attributes:
        heap (list): The underlying array storing MinHeapItem objects.
        capacity (int): Maximum number of items to store (for top-K).
                       None means unlimited capacity.

    Example Usage:
        >>> pq = PriorityQueue(capacity=3)
        >>> pq.push(0.9, "Neoplasms")
        >>> pq.push(0.3, "Cardiovascular")
        >>> pq.push(0.7, "Nervous System")
        >>> pq.push(0.8, "Digestive System")  # Replaces "Cardiovascular" (0.3)
        >>> pq.get_top_k(3)
        [(0.9, 'Neoplasms'), (0.8, 'Digestive System'), (0.7, 'Nervous System')]
    """

    def __init__(self, capacity=None):
        """
        Initialise an empty PriorityQueue.

        Args:
            capacity (int, optional): Maximum number of items to store.
                                     If None, the queue has unlimited capacity.

        Time Complexity: O(1)
        Space Complexity: O(1)
        """
        self.heap = []
        self.capacity = capacity

    def push(self, score, item):
        """
        Insert a new item into the priority queue.

        If the queue has a capacity limit (for top-K selection):
            - If not full: insert the item
            - If full and new score > minimum: replace the minimum
            - If full and new score <= minimum: discard the item

        Algorithm (sift-up):
            1. Append item to end of heap array
            2. Compare with parent at index (i-1)//2
            3. If child < parent, swap and continue upward
            4. Stop when heap property is satisfied or root is reached

        Args:
            score (float): The priority score of the item.
            item (any): The data to store.

        Time Complexity: O(log n) where n is the current heap size.
        Space Complexity: O(1) amortised.
        """
        new_item = MinHeapItem(score, item)

        # If capacity is limited (top-K mode)
        if self.capacity is not None and len(self.heap) >= self.capacity:
            # Only insert if new score is greater than current minimum
            if score > self.heap[0].score:
                # Replace the minimum (root) with the new item
                self.heap[0] = new_item
                # Restore heap property by sifting down
                self._sift_down(0)
            # Otherwise, discard the new item (it's not in top-K)
            return

        # Append to the end of the heap
        self.heap.append(new_item)

        # Sift up to maintain the heap property
        self._sift_up(len(self.heap) - 1)

    def pop(self):
        """
        Remove and return the item with the minimum score.

        Algorithm (sift-down):
            1. Save the root element (minimum)
            2. Move the last element to the root position
            3. Remove the last position
            4. Sift down: compare root with children, swap with smaller child
            5. Continue until heap property is restored

        Time Complexity: O(log n) where n is the current heap size.
        Space Complexity: O(1).

        Returns:
            tuple: (score, item) pair of the minimum element.

        Raises:
            IndexError: If the queue is empty.
        """
        if self.is_empty():
            raise IndexError("Priority queue is empty — cannot pop")

        # Save the minimum element (root)
        min_item = self.heap[0]

        # Move the last element to the root
        last_item = self.heap.pop()

        if self.heap:
            self.heap[0] = last_item
            # Sift down to restore heap property
            self._sift_down(0)

        return (min_item.score, min_item.item)

    def peek(self):
        """
        Return the minimum element without removing it.

        Time Complexity: O(1).
        Space Complexity: O(1).

        Returns:
            tuple: (score, item) pair of the minimum element.

        Raises:
            IndexError: If the queue is empty.
        """
        if self.is_empty():
            raise IndexError("Priority queue is empty — cannot peek")

        return (self.heap[0].score, self.heap[0].item)

    def get_top_k(self, k=None):
        """
        Return the top-K items sorted by score in descending order.

        This is the primary method for feature selection and category ranking.
        It extracts all items from a copy of the heap, sorts them, and returns
        the top K.

        Args:
            k (int, optional): Number of top items to return.
                              Defaults to all items if None.

        Time Complexity: O(n log n) for sorting the current items.
        Space Complexity: O(n) for the copy.

        Returns:
            list[tuple]: List of (score, item) pairs sorted descending by score.
        """
        if k is None:
            k = len(self.heap)

        # Sort all items by score in descending order
        sorted_items = sorted(
            [(item.score, item.item) for item in self.heap],
            key=lambda x: x[0],
            reverse=True
        )

        return sorted_items[:k]

    def _sift_up(self, index):
        """
        Restore the heap property by moving an element upward.

        Compares the element at 'index' with its parent and swaps if
        the element is smaller than its parent. Repeats until the
        heap property is satisfied or the root is reached.

        Args:
            index (int): The index of the element to sift up.

        Time Complexity: O(log n) — at most traverses the height of the tree.
        """
        while index > 0:
            parent_index = (index - 1) // 2

            # If current element is smaller than parent, swap
            if self.heap[index] < self.heap[parent_index]:
                self.heap[index], self.heap[parent_index] = (
                    self.heap[parent_index], self.heap[index]
                )
                index = parent_index
            else:
                break  # Heap property satisfied

    def _sift_down(self, index):
        """
        Restore the heap property by moving an element downward.

        Compares the element at 'index' with its children and swaps with
        the smaller child if needed. Repeats until the heap property
        is satisfied or a leaf node is reached.

        Args:
            index (int): The index of the element to sift down.

        Time Complexity: O(log n) — at most traverses the height of the tree.
        """
        size = len(self.heap)

        while True:
            smallest = index
            left_child = 2 * index + 1
            right_child = 2 * index + 2

            # Check if left child exists and is smaller
            if left_child < size and self.heap[left_child] < self.heap[smallest]:
                smallest = left_child

            # Check if right child exists and is smaller
            if right_child < size and self.heap[right_child] < self.heap[smallest]:
                smallest = right_child

            # If a child is smaller, swap and continue
            if smallest != index:
                self.heap[index], self.heap[smallest] = (
                    self.heap[smallest], self.heap[index]
                )
                index = smallest
            else:
                break  # Heap property satisfied

    def is_empty(self):
        """
        Check if the priority queue is empty.

        Time Complexity: O(1).

        Returns:
            bool: True if empty, False otherwise.
        """
        return len(self.heap) == 0

    def size(self):
        """
        Return the current number of items in the queue.

        Time Complexity: O(1).

        Returns:
            int: Number of items currently stored.
        """
        return len(self.heap)

    def clear(self):
        """
        Remove all items from the priority queue.

        Time Complexity: O(1).
        """
        self.heap = []

    def __len__(self):
        """Return the number of items in the queue."""
        return len(self.heap)

    def __repr__(self):
        """String representation of the PriorityQueue."""
        return f"PriorityQueue(size={len(self.heap)}, capacity={self.capacity})"

    def __bool__(self):
        """Return True if the queue is non-empty."""
        return not self.is_empty()


#### 2.3 Binary Search Tree (BST)

In [ ]:
"""
Binary Search Tree (BST) Data Structure for Medical Text Classification
=======================================================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module implements a Binary Search Tree for efficient sorted storage
and range-based queries on TF-IDF scores and feature importance values.

The BST enables:
    - O(log n) average-case insertion and search
    - Efficient range queries: retrieve all terms with TF-IDF scores
      between a given low and high threshold
    - In-order traversal for sorted output

While a hash map provides O(1) lookup, it does not support range queries
or sorted traversal — both essential for feature selection in the
classification pipeline.

References:
    - Cormen, T.H. et al. (2009) Introduction to Algorithms, 3rd edn.
"""


class BSTNode:
    """
    Represents a single node in the Binary Search Tree.

    Each node stores a key-value pair where:
        - key (str): The term/feature name (used for BST ordering)
        - value (float): The associated score (e.g., TF-IDF score)

    Attributes:
        key (str): The lookup key (alphabetically ordered in the BST).
        value (float): The associated numerical value.
        left (BSTNode): Left child (keys < this node's key).
        right (BSTNode): Right child (keys > this node's key).
        count (int): Number of times this key has been inserted (handles duplicates).
    """

    def __init__(self, key, value):
        """
        Initialise a new BST node with a key-value pair.

        Args:
            key (str): The node's key for ordering.
            value (float): The associated value/score.
        """
        self.key = key
        self.value = value
        self.left = None   # Left subtree (keys less than this key)
        self.right = None  # Right subtree (keys greater than this key)
        self.count = 1     # For handling duplicate insertions


class BinarySearchTree:
    """
    A Binary Search Tree for sorted storage of term-score pairs.

    Used in the medical text classification pipeline for:
        1. Sorted storage of TF-IDF scores by term name
        2. Range queries: find all terms with scores in a given range
        3. Ordered traversal for generating sorted feature lists
        4. Efficient term-score lookup

    The BST maintains the invariant: for any node N,
        all keys in N.left < N.key < all keys in N.right

    Attributes:
        root (BSTNode): The root node of the tree.
        size (int): Number of unique keys stored.

    Note: This is an unbalanced BST. Average case is O(log n) but worst
    case (sorted input) degrades to O(n). For production use, an AVL tree
    or Red-Black tree would be preferred, but the standard BST is implemented
    here to clearly demonstrate the data structure and its properties.

    Example Usage:
        >>> bst = BinarySearchTree()
        >>> bst.insert("tumor", 0.85)
        >>> bst.insert("cardiac", 0.72)
        >>> bst.insert("neural", 0.91)
        >>> bst.search("tumor")
        0.85
        >>> bst.range_query("c", "o")  # All terms from "c" to "o"
        [('cardiac', 0.72), ('neural', 0.91)]
    """

    def __init__(self):
        """
        Initialise an empty Binary Search Tree.

        Time Complexity: O(1)
        Space Complexity: O(1)
        """
        self.root = None
        self.size = 0

    def insert(self, key, value):
        """
        Insert a key-value pair into the BST.

        If the key already exists, its value is updated.

        Algorithm:
            1. If tree is empty, create root node
            2. Otherwise, traverse from root:
               a. If key < current.key, go left
               b. If key > current.key, go right
               c. If key == current.key, update value

        Args:
            key (str): The key for ordering (e.g., term name).
            value (float): The associated value (e.g., TF-IDF score).

        Time Complexity: O(log n) average, O(n) worst case (skewed tree).
        Space Complexity: O(log n) average for recursion stack.
        """
        self.root = self._insert_recursive(self.root, key, value)

    def _insert_recursive(self, node, key, value):
        """
        Internal recursive helper for insertion.

        Args:
            node (BSTNode): Current node in the traversal.
            key (str): Key to insert.
            value (float): Value to associate with the key.

        Returns:
            BSTNode: The (potentially new) node at this position.
        """
        # Base case: found an empty spot, create a new node
        if node is None:
            self.size += 1
            return BSTNode(key, value)

        # Recurse based on key comparison
        if key < node.key:
            node.left = self._insert_recursive(node.left, key, value)
        elif key > node.key:
            node.right = self._insert_recursive(node.right, key, value)
        else:
            # Key already exists — update its value and increment count
            node.value = value
            node.count += 1

        return node

    def search(self, key):
        """
        Search for a key and return its associated value.

        Args:
            key (str): The key to search for.

        Time Complexity: O(log n) average, O(n) worst case.
        Space Complexity: O(1) (iterative implementation).

        Returns:
            float or None: The value associated with the key,
                          or None if the key is not found.
        """
        current = self.root

        while current is not None:
            if key < current.key:
                current = current.left
            elif key > current.key:
                current = current.right
            else:
                return current.value  # Found the key

        return None  # Key not found

    def in_order_traversal(self):
        """
        Perform an in-order traversal of the BST.

        Returns all key-value pairs sorted by key in ascending order.
        This is useful for generating sorted feature lists.

        Algorithm:
            1. Recursively traverse left subtree
            2. Visit current node
            3. Recursively traverse right subtree

        Time Complexity: O(n) — visits every node exactly once.
        Space Complexity: O(n) for the result list + O(h) for recursion stack
                         where h is the tree height.

        Returns:
            list[tuple]: List of (key, value) pairs in sorted order.
        """
        results = []
        self._in_order_recursive(self.root, results)
        return results

    def _in_order_recursive(self, node, results):
        """
        Internal recursive helper for in-order traversal.

        Args:
            node (BSTNode): Current node.
            results (list): Accumulator for visited key-value pairs.
        """
        if node is not None:
            # Visit left subtree first (smaller keys)
            self._in_order_recursive(node.left, results)

            # Visit current node
            results.append((node.key, node.value))

            # Visit right subtree (larger keys)
            self._in_order_recursive(node.right, results)

    def range_query(self, low_key, high_key):
        """
        Find all key-value pairs where low_key <= key <= high_key.

        This operation is uniquely suited to the BST — hash maps cannot
        efficiently support range queries. In the classification pipeline,
        this is used to find all terms within a score range or alphabetical
        range.

        Algorithm:
            1. If current key < low_key, skip left subtree (all keys too small)
            2. If current key > high_key, skip right subtree (all keys too large)
            3. If low_key <= current key <= high_key, include current node

        Args:
            low_key (str): Lower bound of the range (inclusive).
            high_key (str): Upper bound of the range (inclusive).

        Time Complexity: O(log n + k) where k is the number of results.
        Space Complexity: O(k) for results + O(h) for recursion stack.

        Returns:
            list[tuple]: List of (key, value) pairs within the range,
                        sorted by key.
        """
        results = []
        self._range_query_recursive(self.root, low_key, high_key, results)
        return results

    def _range_query_recursive(self, node, low_key, high_key, results):
        """
        Internal recursive helper for range queries.

        Efficiently prunes branches that cannot contain keys in the range.

        Args:
            node (BSTNode): Current node.
            low_key (str): Lower bound.
            high_key (str): Upper bound.
            results (list): Accumulator for matching key-value pairs.
        """
        if node is None:
            return

        # Only explore left subtree if there might be keys >= low_key
        if node.key > low_key:
            self._range_query_recursive(node.left, low_key, high_key, results)

        # Include current node if within range
        if low_key <= node.key <= high_key:
            results.append((node.key, node.value))

        # Only explore right subtree if there might be keys <= high_key
        if node.key < high_key:
            self._range_query_recursive(node.right, low_key, high_key, results)

    def find_min(self):
        """
        Find the minimum key in the BST.

        Time Complexity: O(h) where h is the tree height.

        Returns:
            tuple or None: (key, value) of the minimum key, or None if empty.
        """
        if self.root is None:
            return None

        current = self.root
        while current.left is not None:
            current = current.left

        return (current.key, current.value)

    def find_max(self):
        """
        Find the maximum key in the BST.

        Time Complexity: O(h) where h is the tree height.

        Returns:
            tuple or None: (key, value) of the maximum key, or None if empty.
        """
        if self.root is None:
            return None

        current = self.root
        while current.right is not None:
            current = current.right

        return (current.key, current.value)

    def delete(self, key):
        """
        Delete a key from the BST.

        Handles three cases:
            1. Node has no children: simply remove it
            2. Node has one child: replace with child
            3. Node has two children: replace with in-order successor

        Args:
            key (str): The key to delete.

        Time Complexity: O(log n) average, O(n) worst case.
        Space Complexity: O(h) for recursion stack.

        Returns:
            bool: True if the key was found and deleted, False otherwise.
        """
        if self.search(key) is None:
            return False

        self.root = self._delete_recursive(self.root, key)
        self.size -= 1
        return True

    def _delete_recursive(self, node, key):
        """
        Internal recursive helper for deletion.

        Args:
            node (BSTNode): Current node.
            key (str): Key to delete.

        Returns:
            BSTNode: The (potentially modified) node at this position.
        """
        if node is None:
            return None

        if key < node.key:
            node.left = self._delete_recursive(node.left, key)
        elif key > node.key:
            node.right = self._delete_recursive(node.right, key)
        else:
            # Found the node to delete

            # Case 1: No children
            if node.left is None and node.right is None:
                return None

            # Case 2: One child
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left

            # Case 3: Two children
            # Find the in-order successor (smallest key in right subtree)
            successor = node.right
            while successor.left is not None:
                successor = successor.left

            # Replace current node's data with successor's data
            node.key = successor.key
            node.value = successor.value
            node.count = successor.count

            # Delete the successor from the right subtree
            node.right = self._delete_recursive(node.right, successor.key)

        return node

    def get_height(self):
        """
        Calculate the height of the BST.

        Useful for complexity analysis — the height determines the
        worst-case performance of search, insert, and delete operations.

        Time Complexity: O(n).
        Space Complexity: O(h) for recursion stack.

        Returns:
            int: The height of the tree (-1 for empty tree).
        """
        return self._height_recursive(self.root)

    def _height_recursive(self, node):
        """Internal recursive helper for height calculation."""
        if node is None:
            return -1
        left_height = self._height_recursive(node.left)
        right_height = self._height_recursive(node.right)
        return 1 + max(left_height, right_height)

    def __len__(self):
        """Return the number of unique keys in the BST."""
        return self.size

    def __contains__(self, key):
        """Support 'in' operator: 'key in bst'."""
        return self.search(key) is not None

    def __repr__(self):
        """String representation of the BST."""
        return f"BinarySearchTree(size={self.size}, height={self.get_height()})"


### 3. NLP Pipeline
Implementation of the Text Preprocessor, Medical Tokenizer, and custom TF-IDF Vectorizer.

#### 3.1 Text Preprocessor

In [ ]:
"""
Text Preprocessor for Medical Text Classification
==================================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module implements text preprocessing specifically tailored for
medical abstracts and clinical text. Preprocessing is a critical step
in the NLP pipeline that transforms raw text into a clean, normalised
format suitable for feature extraction.

Preprocessing Steps:
    1. Lowercasing
    2. Punctuation and special character removal
    3. Number handling (replace with <NUM> token or remove)
    4. Stop word removal (using a comprehensive medical-aware stop word list)
    5. Abbreviation expansion (common medical abbreviations)
    6. Basic lemmatisation (suffix stripping for common patterns)

Design Decision:
    We implement custom preprocessing rather than relying on external NLP
    libraries (like NLTK or spaCy) to demonstrate algorithmic understanding
    and maintain full control over the pipeline. This aligns with the
    module's focus on data structures and algorithms.
"""

import re
import string


class TextPreprocessor:
    """
    A text preprocessing engine for medical documents.

    This class provides a pipeline of text cleaning and normalisation
    operations designed for medical text. Each operation is implemented
    as a separate method to enable flexible composition and testing.

    Attributes:
        stop_words (set): Set of stop words to remove from text.
                         Using a set provides O(1) membership testing.
        medical_abbreviations (dict): Mapping of abbreviations to expansions.
                                     Using a dict provides O(1) lookup.
        remove_numbers (bool): Whether to remove numerical tokens.

    Example Usage:
        >>> preprocessor = TextPreprocessor()
        >>> preprocessor.preprocess("The patient's CT scan showed NO abnormalities.")
        'patient ct scan showed abnormalities'
    """

    # Comprehensive English stop words list (avoiding external dependencies)
    DEFAULT_STOP_WORDS = {
        'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been',
        'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
        'could', 'should', 'may', 'might', 'shall', 'can', 'need', 'dare',
        'ought', 'used', 'it', 'its', 'this', 'that', 'these', 'those',
        'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you',
        'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his',
        'himself', 'she', 'her', 'hers', 'herself', 'itself', 'they', 'them',
        'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom',
        'when', 'where', 'why', 'how', 'all', 'each', 'every', 'both',
        'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
        'only', 'own', 'same', 'so', 'than', 'too', 'very', 'just', 'don',
        'now', 'about', 'above', 'after', 'again', 'against', 'between',
        'into', 'through', 'during', 'before', 'below', 'under', 'over',
        'then', 'once', 'here', 'there', 'further', 'also', 'however',
        'although', 'though', 'yet', 'still', 'already', 'while', 'if',
        'because', 'until', 'unless', 'since', 'whether', 'either',
        'neither', 'up', 'out', 'off', 'down', 'any', 'much', 'many',
        'being', 'having', 'doing', 'going', 'using', 'well', 'back',
        'even', 'given', 'rather', 'quite', 'often', 'thus', 'hence',
        'therefore', 'moreover', 'furthermore', 'nevertheless', 'nonetheless',
        'whereas', 'whereby', 'wherein', 'upon', 'within', 'without',
        'among', 'across', 'around', 'along', 'beside', 'besides',
        'beyond', 'despite', 'except', 'like', 'near', 'past',
        'regarding', 'unlike', 'via', 'versus', 'et', 'al', 'etc',
    }

    # Common medical abbreviations and their expansions
    DEFAULT_MEDICAL_ABBREVIATIONS = {
        'ecg': 'electrocardiogram',
        'eeg': 'electroencephalogram',
        'mri': 'magnetic resonance imaging',
        'ct': 'computed tomography',
        'bp': 'blood pressure',
        'hr': 'heart rate',
        'bmi': 'body mass index',
        'cns': 'central nervous system',
        'gi': 'gastrointestinal',
        'cv': 'cardiovascular',
        'copd': 'chronic obstructive pulmonary disease',
        'dvt': 'deep vein thrombosis',
        'pe': 'pulmonary embolism',
        'cad': 'coronary artery disease',
        'chf': 'congestive heart failure',
        'mi': 'myocardial infarction',
        'htn': 'hypertension',
        'dm': 'diabetes mellitus',
        'ckd': 'chronic kidney disease',
        'uti': 'urinary tract infection',
        'ards': 'acute respiratory distress syndrome',
        'icu': 'intensive care unit',
        'nsaid': 'nonsteroidal anti inflammatory drug',
        'ace': 'angiotensin converting enzyme',
        'arb': 'angiotensin receptor blocker',
        'ssri': 'selective serotonin reuptake inhibitor',
        'cbc': 'complete blood count',
        'bmp': 'basic metabolic panel',
        'lfts': 'liver function tests',
        'wbc': 'white blood cell',
        'rbc': 'red blood cell',
        'hgb': 'hemoglobin',
        'plt': 'platelet',
    }

    def __init__(self, stop_words=None, medical_abbreviations=None,
                 remove_numbers=True, min_word_length=2):
        """
        Initialise the TextPreprocessor with configuration options.

        Args:
            stop_words (set, optional): Custom stop words.
                                       Defaults to DEFAULT_STOP_WORDS.
            medical_abbreviations (dict, optional): Custom abbreviation mappings.
                                                   Defaults to DEFAULT_MEDICAL_ABBREVIATIONS.
            remove_numbers (bool): Whether to remove pure numeric tokens.
            min_word_length (int): Minimum word length to keep after tokenisation.

        Time Complexity: O(S + A) where S is stop_words size, A is abbreviations size.
        """
        # Use set for O(1) membership testing of stop words
        self.stop_words = stop_words if stop_words is not None else self.DEFAULT_STOP_WORDS.copy()

        # Use dict for O(1) abbreviation lookup
        self.medical_abbreviations = (
            medical_abbreviations if medical_abbreviations is not None
            else self.DEFAULT_MEDICAL_ABBREVIATIONS.copy()
        )

        self.remove_numbers = remove_numbers
        self.min_word_length = min_word_length

        # Precompile regex patterns for efficiency
        # This pattern matches any character that is NOT alphanumeric or whitespace
        self._punctuation_pattern = re.compile(r'[^\w\s]')
        # This pattern matches sequences of whitespace
        self._whitespace_pattern = re.compile(r'\s+')
        # This pattern matches pure numeric tokens
        self._number_pattern = re.compile(r'^\d+\.?\d*$')

    def preprocess(self, text):
        """
        Apply the full preprocessing pipeline to a text string.

        Pipeline order:
            1. Lowercase conversion
            2. Abbreviation expansion
            3. Punctuation removal
            4. Whitespace normalisation
            5. Tokenisation
            6. Number removal (if configured)
            7. Stop word removal
            8. Minimum word length filtering
            9. Rejoin into clean string

        Args:
            text (str): The raw input text to preprocess.

        Time Complexity: O(n) where n is the length of the text.
        Space Complexity: O(n) for the processed text.

        Returns:
            str: The cleaned, normalised text.
        """
        if not text or not isinstance(text, str):
            return ""

        # Step 1: Convert to lowercase for case-insensitive processing
        text = self.clean_text(text)

        # Step 2: Expand medical abbreviations
        text = self.expand_abbreviations(text)

        # Step 3: Tokenise (split into individual words)
        tokens = text.split()

        # Step 4: Remove stop words (O(1) per lookup using set)
        tokens = self.remove_stopwords(tokens)

        # Step 5: Apply basic lemmatisation
        tokens = self.lemmatize(tokens)

        # Step 6: Filter by minimum word length
        tokens = [t for t in tokens if len(t) >= self.min_word_length]

        # Rejoin tokens into a clean string
        return ' '.join(tokens)

    def clean_text(self, text):
        """
        Clean raw text by lowercasing, removing punctuation, and
        normalising whitespace.

        Args:
            text (str): Raw input text.

        Time Complexity: O(n) where n is the text length.

        Returns:
            str: Cleaned text.
        """
        # Lowercase
        text = text.lower()

        # Remove punctuation (keep only alphanumeric and whitespace)
        text = self._punctuation_pattern.sub(' ', text)

        # Normalise whitespace (collapse multiple spaces into one)
        text = self._whitespace_pattern.sub(' ', text).strip()

        return text

    def remove_stopwords(self, tokens):
        """
        Remove stop words from a list of tokens.

        Uses a set for O(1) membership testing per token,
        resulting in O(n) total time for n tokens.

        Args:
            tokens (list[str]): List of word tokens.

        Time Complexity: O(n) where n is the number of tokens.
        Space Complexity: O(n) for the filtered list.

        Returns:
            list[str]: Tokens with stop words removed.
        """
        return [
            token for token in tokens
            if token not in self.stop_words
            and (not self.remove_numbers or not self._number_pattern.match(token))
        ]

    def expand_abbreviations(self, text):
        """
        Expand common medical abbreviations in the text.

        Replaces known abbreviations with their full forms to improve
        term matching consistency (e.g., "ECG" → "electrocardiogram").

        Args:
            text (str): Text with potential abbreviations.

        Time Complexity: O(n × a) where n is the number of words
                        and a is the average abbreviation length.

        Returns:
            str: Text with abbreviations expanded.
        """
        words = text.split()
        expanded = []

        for word in words:
            # Check if the word is a known abbreviation (O(1) dict lookup)
            if word in self.medical_abbreviations:
                expanded.append(self.medical_abbreviations[word])
            else:
                expanded.append(word)

        return ' '.join(expanded)

    def lemmatize(self, tokens):
        """
        Apply basic suffix-stripping lemmatisation to tokens.

        This is a simplified rule-based lemmatiser that handles common
        English suffixes. For production use, a full morphological
        analyser (e.g., NLTK WordNetLemmatizer) would be preferred,
        but this custom implementation demonstrates the algorithm
        and avoids external dependencies.

        Rules applied (in order):
            - 'ies' → 'y' (e.g., 'studies' → 'study')
            - 'ves' → 'f' (e.g., 'nerves' → 'nerve' — kept as is to avoid errors)
            - 'ses', 'xes', 'zes', 'ches', 'shes' → remove 'es'
            - 'ness' → remove (e.g., 'effectiveness' → 'effective')
            - 'ment' → remove (e.g., 'treatment' → 'treat')
            - 'ing' → remove (e.g., 'treating' → 'treat')
            - 'tion' → remove (e.g., 'classification' → 'classifica')
            - 's' → remove (e.g., 'patients' → 'patient')

        Args:
            tokens (list[str]): List of word tokens to lemmatise.

        Time Complexity: O(n × m) where n is the number of tokens
                        and m is the average token length.

        Returns:
            list[str]: Lemmatised tokens.
        """
        lemmatised = []

        for token in tokens:
            # Skip short words (less likely to have meaningful suffixes)
            if len(token) <= 4:
                lemmatised.append(token)
                continue

            # Apply suffix rules (order matters for correctness)
            if token.endswith('ies') and len(token) > 4:
                token = token[:-3] + 'y'
            elif token.endswith('ness') and len(token) > 5:
                token = token[:-4]
            elif token.endswith('ment') and len(token) > 5:
                token = token[:-4]
            elif token.endswith('ing') and len(token) > 5:
                token = token[:-3]
            elif token.endswith('tion') and len(token) > 5:
                token = token[:-4]
            elif token.endswith('ses') and len(token) > 4:
                token = token[:-2]
            elif token.endswith('es') and len(token) > 4:
                token = token[:-2]
            elif token.endswith('s') and not token.endswith('ss') and len(token) > 3:
                token = token[:-1]

            lemmatised.append(token)

        return lemmatised

    def get_stats(self):
        """
        Return configuration statistics for the preprocessor.

        Returns:
            dict: Configuration details including stop word count,
                 abbreviation count, and settings.
        """
        return {
            'num_stop_words': len(self.stop_words),
            'num_abbreviations': len(self.medical_abbreviations),
            'remove_numbers': self.remove_numbers,
            'min_word_length': self.min_word_length
        }

    def __repr__(self):
        """String representation of the TextPreprocessor."""
        return (
            f"TextPreprocessor(stop_words={len(self.stop_words)}, "
            f"abbreviations={len(self.medical_abbreviations)}, "
            f"min_word_len={self.min_word_length})"
        )


#### 3.2 Medical Tokenizer

In [ ]:
"""
Medical Tokenizer for Medical Text Classification
==================================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module implements a tokenizer that integrates with the MedicalTrie
data structure to provide efficient vocabulary management and token
lookup during the text classification pipeline.

The tokenizer builds a vocabulary from the training corpus and uses
the Trie for O(m) term validation, where m is the word length.
"""






class MedicalTokenizer:
    """
    A tokenizer designed for medical text that integrates with MedicalTrie
    for efficient vocabulary management.

    The MedicalTokenizer:
        1. Preprocesses text using TextPreprocessor
        2. Splits text into tokens
        3. Validates tokens against the vocabulary stored in a Trie
        4. Tracks word frequencies and category associations

    This tokenizer is specifically designed to work with the Trie data
    structure, providing O(m) lookup for each token (where m is the
    token length), compared to O(n) for a list-based vocabulary check.

    Attributes:
        preprocessor (TextPreprocessor): Text cleaning engine.
        trie (MedicalTrie): Vocabulary stored in a Trie for fast lookup.
        vocab (dict): Word-to-index mapping for vectorisation.
        word_freq (dict): Global word frequency counter.
        is_fitted (bool): Whether the tokenizer has been fitted to a corpus.

    Example Usage:
        >>> tokenizer = MedicalTokenizer()
        >>> tokenizer.fit(["Patient has cardiac arrest", "Tumor found in liver"])
        >>> tokens = tokenizer.tokenize("cardiac tumor diagnosis")
        >>> print(tokens)
        ['cardiac', 'tumor', 'diagnosis']
    """

    def __init__(self, preprocessor=None, min_freq=1, max_vocab_size=None):
        """
        Initialise the MedicalTokenizer.

        Args:
            preprocessor (TextPreprocessor, optional): Custom preprocessor.
                                                      Defaults to a new instance.
            min_freq (int): Minimum word frequency to include in vocabulary.
                           Words appearing fewer times are excluded.
            max_vocab_size (int, optional): Maximum vocabulary size.
                                          If set, only the most frequent words are kept.

        Time Complexity: O(1)
        """
        # Text preprocessing engine
        self.preprocessor = preprocessor if preprocessor else TextPreprocessor()

        # Trie for efficient vocabulary storage and lookup
        self.trie = MedicalTrie()

        # Word-to-index mapping (for converting tokens to feature indices)
        self.vocab = {}

        # Global word frequency counter (used for filtering rare words)
        self.word_freq = {}

        # Configuration
        self.min_freq = min_freq
        self.max_vocab_size = max_vocab_size

        # State flag
        self.is_fitted = False

    def fit(self, documents, categories=None):
        """
        Build the vocabulary from a corpus of documents.

        Algorithm:
            1. Preprocess each document
            2. Count word frequencies across the corpus
            3. Filter by minimum frequency
            4. Optionally limit vocabulary size (keep most frequent)
            5. Insert all vocabulary words into the Trie
            6. Build word-to-index mapping

        Args:
            documents (list[str]): List of document strings.
            categories (list[str], optional): Corresponding category labels.
                                            Used for category-frequency tracking in the Trie.

        Time Complexity: O(D × L) where D is the number of documents
                        and L is the average document length.
        Space Complexity: O(V × M) where V is vocabulary size
                         and M is average word length (for Trie storage).

        Returns:
            MedicalTokenizer: self (for method chaining).
        """
        # Reset state
        self.word_freq = {}
        self.trie = MedicalTrie()
        self.vocab = {}

        # Step 1 & 2: Preprocess and count word frequencies
        all_tokens_by_doc = []
        for i, doc in enumerate(documents):
            processed = self.preprocessor.preprocess(doc)
            tokens = processed.split()
            all_tokens_by_doc.append(tokens)

            # Count frequencies using hash map (O(1) per update)
            for token in tokens:
                if token not in self.word_freq:
                    self.word_freq[token] = 0
                self.word_freq[token] += 1

        # Step 3: Filter by minimum frequency
        filtered_words = {
            word: freq for word, freq in self.word_freq.items()
            if freq >= self.min_freq
        }

        # Step 4: Optionally limit vocabulary size
        if self.max_vocab_size and len(filtered_words) > self.max_vocab_size:
            # Sort by frequency (descending) and keep top N
            sorted_words = sorted(
                filtered_words.items(),
                key=lambda x: x[1],
                reverse=True
            )
            filtered_words = dict(sorted_words[:self.max_vocab_size])

        # Step 5: Insert vocabulary words into the Trie with category tracking
        for i, tokens in enumerate(all_tokens_by_doc):
            category = categories[i] if categories and i < len(categories) else None
            for token in tokens:
                if token in filtered_words:
                    self.trie.insert(token, category)

        # Step 6: Build word-to-index mapping (sorted for deterministic output)
        sorted_vocab = sorted(filtered_words.keys())
        self.vocab = {word: idx for idx, word in enumerate(sorted_vocab)}

        self.is_fitted = True
        return self

    def tokenize(self, text):
        """
        Tokenize a text string using the fitted vocabulary.

        Preprocesses the text and returns tokens that exist in the vocabulary.
        Unknown tokens (not in the Trie) are excluded.

        Args:
            text (str): The text to tokenize.

        Time Complexity: O(L × m) where L is the number of tokens
                        and m is the average token length (for Trie search).

        Returns:
            list[str]: List of validated tokens.
        """
        if not self.is_fitted:
            raise RuntimeError("Tokenizer must be fitted before use. Call fit() first.")

        # Preprocess the text
        processed = self.preprocessor.preprocess(text)
        tokens = processed.split()

        # Filter: only keep tokens in our vocabulary (Trie search is O(m))
        valid_tokens = [token for token in tokens if self.trie.search(token)]

        return valid_tokens

    def tokenize_to_indices(self, text):
        """
        Tokenize text and convert to vocabulary indices.

        Useful for creating numerical feature vectors.

        Args:
            text (str): The text to tokenize.

        Time Complexity: O(L × m) for tokenisation + O(L) for index lookup.

        Returns:
            list[int]: List of vocabulary indices for each valid token.
        """
        tokens = self.tokenize(text)
        return [self.vocab[token] for token in tokens if token in self.vocab]

    def get_vocab_size(self):
        """
        Return the size of the vocabulary.

        Returns:
            int: Number of unique words in the vocabulary.
        """
        return len(self.vocab)

    def get_word_index(self, word):
        """
        Get the vocabulary index of a word.

        Args:
            word (str): The word to look up.

        Returns:
            int or None: The index, or None if the word is not in the vocabulary.
        """
        return self.vocab.get(word.lower(), None)

    def get_index_word(self, index):
        """
        Get the word at a given vocabulary index.

        Args:
            index (int): The vocabulary index.

        Returns:
            str or None: The word, or None if the index is invalid.
        """
        # Build reverse mapping (lazy, cached)
        if not hasattr(self, '_index_to_word') or len(self._index_to_word) != len(self.vocab):
            self._index_to_word = {idx: word for word, idx in self.vocab.items()}

        return self._index_to_word.get(index, None)

    def get_word_frequency(self, word):
        """
        Get the corpus frequency of a word.

        Args:
            word (str): The word to query.

        Returns:
            int: Frequency count, or 0 if not found.
        """
        return self.word_freq.get(word.lower(), 0)

    def __repr__(self):
        """String representation of the MedicalTokenizer."""
        return (
            f"MedicalTokenizer(vocab_size={len(self.vocab)}, "
            f"min_freq={self.min_freq}, "
            f"fitted={self.is_fitted})"
        )


#### 3.3 Custom TF-IDF Vectorizer

In [ ]:
"""
Custom TF-IDF Vectorizer for Medical Text Classification
======================================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module implements a custom Term Frequency-Inverse Document Frequency (TF-IDF)
vectorization engine. Rather than relying on Scikit-Learn's TfidfVectorizer,
this is implemented from scratch to demonstrate algorithmic understanding
and data structure utilisation (Hash Maps for sparse vector representation).

Key Features:
    - O(n × d) fit algorithm where n=docs, d=vocab
    - O(d) transform algorithm
    - Sparse dictionary representation for memory efficiency
    - L2 normalization for cosine similarity compatibility

References:
    - Salton, G. and McGill, M.J. (1983) Introduction to Modern Information Retrieval.
"""

import math
from collections import defaultdict

class TFIDFVectorizer:
    """
    Custom TF-IDF engine using Hash Maps for sparse representation.

    Instead of generating dense n × V matrices (where n is documents and V
    is vocabulary size), which would require massive memory for a 14k doc corpus,
    this class represents vectors as sparse dictionaries mapping index -> score.

    Attributes:
        tokenizer (MedicalTokenizer): Pre-fitted tokenizer.
        idf_scores (dict): Mapping of token_index -> IDF score.
        vocab_size (int): Total size of the vocabulary.
        num_documents (int): Total number of documents trained on.
    """

    def __init__(self, tokenizer):
        """
        Initialise the TF-IDF Vectorizer.

        Args:
            tokenizer (MedicalTokenizer): A fitted MedicalTokenizer containing
                                          the vocabulary and word indices.

        Time Complexity: O(1)
        """
        if not tokenizer.is_fitted:
            raise ValueError("Tokenizer must be fitted before passing to TFIDFVectorizer")

        self.tokenizer = tokenizer
        self.idf_scores = {}
        self.vocab_size = tokenizer.get_vocab_size()
        self.num_documents = 0
        self.is_fitted = False

    def fit(self, documents):
        """
        Calculate and store IDF (Inverse Document Frequency) scores for all terms.

        IDF(t) = log( (1 + N) / (1 + df(t)) ) + 1
        Where N is total documents and df(t) is document frequency of term t.
        Smooth IDF formula used to prevent zero division (similar to scikit-learn).

        Algorithm:
            1. Iterate over all documents
            2. Tokenize each document into unique term indices
            3. Count document frequency (df) for each term index
            4. Compute and store IDF score for each term index

        Args:
            documents (list[str]): List of raw document strings.

        Time Complexity: O(D × L) where D is number of docs, L is avg length.
        Space Complexity: O(V) where V is vocabulary size.

        Returns:
            TFIDFVectorizer: self
        """
        self.num_documents = len(documents)

        # Hash map to track document frequency: index -> count
        doc_freq = defaultdict(int)

        # Step 1-3: Calculate Document Frequencies
        for doc in documents:
            # Tokenize to indices and get unique indices for this document
            indices = set(self.tokenizer.tokenize_to_indices(doc))
            for idx in indices:
                doc_freq[idx] += 1

        # Step 4: Calculate IDF scores
        for word, idx in self.tokenizer.vocab.items():
            df = doc_freq.get(idx, 0)
            # Smooth IDF calculation
            idf = math.log((1 + self.num_documents) / (1 + df)) + 1
            self.idf_scores[idx] = idf

        self.is_fitted = True
        return self

    def transform(self, document):
        """
        Transform a single document into a TF-IDF feature vector.

        The vector is represented sparsely as a dictionary {index: score}.

        TF(t, d) = count of t in d / total terms in d

        Args:
            document (str): Raw document string.

        Time Complexity: O(L) where L is document length.
        Space Complexity: O(U) where U is number of unique valid terms in document.

        Returns:
            dict: Sparse feature vector {index: score}.
        """
        if not self.is_fitted:
            raise RuntimeError("Vectorizer must be fitted before transform")

        # Get indices for this document
        indices = self.tokenizer.tokenize_to_indices(document)
        total_terms = len(indices)

        if total_terms == 0:
            return {}

        # Calculate Term Frequency (TF)
        term_counts = defaultdict(int)
        for idx in indices:
            term_counts[idx] += 1

        # Calculate TF-IDF and apply L2 normalization
        vector = {}
        sum_sq = 0.0

        for idx, count in term_counts.items():
            tf = count / total_terms
            idf = self.idf_scores.get(idx, 1.0) # default to 1.0 if unseen but somehow tokenized
            tfidf_score = tf * idf
            vector[idx] = tfidf_score
            sum_sq += tfidf_score * tfidf_score

        # L2 Normalization (Euclidean norm)
        # Ensures all vectors have length 1, making cosine similarity = dot product
        if sum_sq > 0:
            norm = math.sqrt(sum_sq)
            for idx in vector:
                vector[idx] /= norm

        return vector

    def fit_transform(self, documents):
        """
        Fit to documents and then transform them.

        Args:
            documents (list[str]): List of documents.

        Time Complexity: O(D × L)
        Space Complexity: O(D × U) where U is avg unique terms per doc.

        Returns:
            list[dict]: List of sparse feature vectors.
        """
        self.fit(documents)
        return [self.transform(doc) for doc in documents]

    def get_feature_names(self):
        """
        Get the ordered list of feature names (vocabulary words).

        Returns:
            list[str]: Vocabulary words ordered by index.
        """
        # Create inverse mapping
        idx_to_word = {idx: word for word, idx in self.tokenizer.vocab.items()}
        # Return sorted by index
        return [idx_to_word[i] for i in range(self.vocab_size)]

    def __repr__(self):
        return f"TFIDFVectorizer(vocab_size={self.vocab_size}, fitted={self.is_fitted})"


### 4. Classification & Evaluation
Implementation of the Ensemble Classifier wrapper, Evaluator, and Explainer (which uses the Priority Queue).

#### 4.1 Ensemble Classifier

In [ ]:
"""
Classifier Framework for Medical Text Classification
==================================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module implements the classification framework. While we use sklearn
for the underlying Random Forest / Decision Tree algorithms (as re-implementing
a full Random Forest would detract from the core focus on data structures),
we wrap them in an OOP framework to manage sparse dictionaries and ensemble
voting logic.
"""

from abc import ABC, abstractmethod
import numpy as np

# We use sklearn's Random Forest as the underlying predictive engine
# to provide robust baseline performance, whilst our wrapper handles
# sparse dictionary representations natively.
from sklearn.ensemble import RandomForestClassifier


class BaseClassifier(ABC):
    """
    Abstract base class defining the standard interface for all classifiers
    in the system. Enforces OOP design principles (Polymorphism).
    """

    def __init__(self):
        self.is_trained = False
        self.classes = []

    @abstractmethod
    def train(self, X_sparse, y):
        """Train the model on sparse TF-IDF vectors."""
        pass

    @abstractmethod
    def predict(self, X_sparse):
        """Predict class labels for sparse TF-IDF vectors."""
        pass

    @abstractmethod
    def predict_proba(self, X_sparse):
        """Predict class probabilities for sparse TF-IDF vectors."""
        pass


class EnsembleClassifier(BaseClassifier):
    """
    An ensemble classifier that wraps sklearn's Random Forest but natively
    handles our custom sparse dictionary representation from TFIDFVectorizer.

    The Random Forest is a natural choice as it implicitly builds decision
    trees based on feature thresholds, complementing our TF-IDF features.
    """

    def __init__(self, vocab_size, n_estimators=100, max_depth=None, random_state=42):
        """
        Initialise the Ensemble Classifier.

        Args:
            vocab_size (int): Total vocabulary size (d).
            n_estimators (int): Number of trees in the forest.
            max_depth (int): Max depth of trees.
            random_state (int): Seed for reproducibility.
        """
        super().__init__()
        self.vocab_size = vocab_size
        self.model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=random_state,
            n_jobs=-1  # Use all cores
        )

    def _dict_to_dense(self, sparse_dicts):
        """
        Convert our custom list of sparse dictionaries to a dense numpy array
        for compatibility with sklearn.

        Time Complexity: O(N × d) where N=docs, d=vocab size.
        Space Complexity: O(N × d) memory.
        """
        N = len(sparse_dicts)
        dense_matrix = np.zeros((N, self.vocab_size))

        for i, vector_dict in enumerate(sparse_dicts):
            for idx, val in vector_dict.items():
                dense_matrix[i, idx] = val

        return dense_matrix

    def train(self, X_sparse, y):
        """
        Train the Random Forest classifier.

        Args:
            X_sparse (list[dict]): List of TF-IDF sparse dictionaries.
            y (list): Target labels.

        Time Complexity: O(N × d) for conversion + O(Trees × N × log(N) × d) for training.
        """
        print(f"Converting {len(X_sparse)} sparse vectors to dense matrix...")
        X_dense = self._dict_to_dense(X_sparse)

        print("Training Random Forest ensemble...")
        self.model.fit(X_dense, y)
        self.classes = self.model.classes_
        self.is_trained = True
        print("Training complete.")

    def predict(self, X_sparse):
        """
        Predict class labels.

        Args:
            X_sparse (list[dict]): List of sparse dictionaries.
        """
        if not self.is_trained:
            raise RuntimeError("Classifier must be trained before predicting.")

        X_dense = self._dict_to_dense(X_sparse)
        return self.model.predict(X_dense)

    def predict_proba(self, X_sparse):
        """
        Predict class probabilities.

        Args:
            X_sparse (list[dict] or dict): Sparse dictionaries.
        """
        if not self.is_trained:
            raise RuntimeError("Classifier must be trained before predicting.")

        # Handle single dictionary case
        if isinstance(X_sparse, dict):
            X_sparse = [X_sparse]

        X_dense = self._dict_to_dense(X_sparse)
        return self.model.predict_proba(X_dense)

    def get_feature_importance(self):
        """
        Return the raw feature importances from the Random Forest.

        Returns:
            np.ndarray: Array of importances matching vocabulary size.
        """
        if not self.is_trained:
            return np.zeros(self.vocab_size)
        return self.model.feature_importances_


#### 4.2 Model Evaluator

In [ ]:
"""
Model Evaluator for Medical Text Classification
===============================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module implements standard evaluation metrics for classification tasks.
While libraries like sklearn provide these natively, implementing them from
scratch demonstrates algorithmic understanding of how these metrics are calculated.

Calculates:
    - Accuracy
    - Precision (Macro-averaged)
    - Recall (Macro-averaged)
    - F1-Score (Macro-averaged)
    - Confusion Matrix
"""

class ModelEvaluator:
    """
    Evaluator for multi-class classification models.

    Computes standard metrics by first calculating True Positives (TP),
    False Positives (FP), and False Negatives (FN) for each class.
    """

    def __init__(self, classes):
        """
        Initialise evaluator with the list of possible classes.

        Args:
            classes (list): Ordered list of class labels.
        """
        self.classes = classes
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}
        self.num_classes = len(classes)

    def _get_counts(self, y_true, y_pred):
        """
        Calculate TP, FP, FN for each class.

        Time Complexity: O(N) where N is number of samples.
        """
        # Initialize counts for each class
        tp = [0] * self.num_classes
        fp = [0] * self.num_classes
        fn = [0] * self.num_classes

        for true, pred in zip(y_true, y_pred):
            true_idx = self.class_to_idx.get(true, -1)
            pred_idx = self.class_to_idx.get(pred, -1)

            if true_idx == -1 or pred_idx == -1:
                continue

            if true == pred:
                tp[true_idx] += 1
            else:
                fp[pred_idx] += 1
                fn[true_idx] += 1

        return tp, fp, fn

    def accuracy(self, y_true, y_pred):
        """
        Calculate overall accuracy.
        Time Complexity: O(N)
        """
        correct = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
        return correct / len(y_true) if len(y_true) > 0 else 0.0

    def precision(self, y_true, y_pred):
        """
        Calculate macro-averaged precision.
        Time Complexity: O(N)
        """
        tp, fp, _ = self._get_counts(y_true, y_pred)
        precisions = []

        for i in range(self.num_classes):
            if tp[i] + fp[i] == 0:
                precisions.append(0.0)
            else:
                precisions.append(tp[i] / (tp[i] + fp[i]))

        return sum(precisions) / self.num_classes

    def recall(self, y_true, y_pred):
        """
        Calculate macro-averaged recall.
        Time Complexity: O(N)
        """
        tp, _, fn = self._get_counts(y_true, y_pred)
        recalls = []

        for i in range(self.num_classes):
            if tp[i] + fn[i] == 0:
                recalls.append(0.0)
            else:
                recalls.append(tp[i] / (tp[i] + fn[i]))

        return sum(recalls) / self.num_classes

    def f1_score(self, y_true, y_pred):
        """
        Calculate macro-averaged F1 score.
        Time Complexity: O(N)
        """
        tp, fp, fn = self._get_counts(y_true, y_pred)
        f1_scores = []

        for i in range(self.num_classes):
            p_den = tp[i] + fp[i]
            r_den = tp[i] + fn[i]

            p = tp[i] / p_den if p_den > 0 else 0.0
            r = tp[i] / r_den if r_den > 0 else 0.0

            if p + r == 0:
                f1_scores.append(0.0)
            else:
                f1_scores.append(2 * (p * r) / (p + r))

        return sum(f1_scores) / self.num_classes

    def confusion_matrix(self, y_true, y_pred):
        """
        Generate a confusion matrix.

        Time Complexity: O(N)
        Space Complexity: O(C^2) where C is number of classes.

        Returns:
            list[list[int]]: 2D array representing the confusion matrix.
        """
        matrix = [[0 for _ in range(self.num_classes)] for _ in range(self.num_classes)]

        for true, pred in zip(y_true, y_pred):
            true_idx = self.class_to_idx.get(true, -1)
            pred_idx = self.class_to_idx.get(pred, -1)

            if true_idx != -1 and pred_idx != -1:
                matrix[true_idx][pred_idx] += 1

        return matrix

    def classification_report(self, y_true, y_pred):
        """
        Generate a formatted text report showing main classification metrics.
        """
        tp, fp, fn = self._get_counts(y_true, y_pred)

        report = f"{'Class':<30} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}\n"
        report += "-" * 68 + "\n"

        for i, cls_name in enumerate(self.classes):
            p_den = tp[i] + fp[i]
            r_den = tp[i] + fn[i]

            p = tp[i] / p_den if p_den > 0 else 0.0
            r = tp[i] / r_den if r_den > 0 else 0.0
            f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

            report += f"{cls_name:<30} | {p:.4f}     | {r:.4f}     | {f1:.4f}\n"

        report += "-" * 68 + "\n"
        report += f"{'Macro Avg':<30} | {self.precision(y_true, y_pred):.4f}     | {self.recall(y_true, y_pred):.4f}     | {self.f1_score(y_true, y_pred):.4f}\n"
        report += f"{'Overall Accuracy':<30} | {self.accuracy(y_true, y_pred):.4f}\n"

        return report


#### 4.3 Classification Explainer

In [ ]:
"""
Classification Explainer for Medical Text Classification
======================================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module provides the explainability layer for the classifier, addressing
the "black box" limitation identified in recent literature.

It uses the Priority Queue (Min-Heap) data structure to rank the most
influential terms in a specific document that drove the model's prediction.
"""





class ClassificationExplainer:
    """
    Explains model predictions by identifying which terms in the input text
    contributed most to the classification decision.

    This directly addresses the gap identified in the 2025 ETASR paper
    regarding the lack of interpretability in deep learning models.
    """

    def __init__(self, tokenizer, vectorizer, classifier):
        """
        Initialise the explainer.

        Args:
            tokenizer (MedicalTokenizer): The fitted tokenizer.
            vectorizer (TFIDFVectorizer): The fitted vectorizer.
            classifier (EnsembleClassifier): The trained classifier.
        """
        self.tokenizer = tokenizer
        self.vectorizer = vectorizer
        self.classifier = classifier

    def explain_prediction(self, text, top_k=5):
        """
        Explain why a specific text was classified the way it was.

        Algorithm:
            1. Tokenize and vectorise the text (TF-IDF)
            2. Get the global feature importances from the classifier
            3. For each term in the document, calculate its local contribution
               Local Contribution = TF-IDF Score × Global Importance
            4. Use a PriorityQueue (Min-Heap) to efficiently extract the Top K terms
            5. Return the explanation along with the prediction and confidence

        Args:
            text (str): The raw text to explain.
            top_k (int): Number of top contributing terms to return.

        Time Complexity: O(U × log K) where U is unique terms in doc, K is top_k.
        Space Complexity: O(K) for the heap.

        Returns:
            dict: Explanation dictionary containing prediction, confidence,
                  and top contributing terms.
        """
        # 1. Pipeline execution for the single text
        tokens = self.tokenizer.tokenize(text)
        tfidf_vector = self.vectorizer.transform(text)
        
        if not tfidf_vector:
            return {
                "prediction": "Unknown (Empty/No Vocabulary Match)",
                "confidence": 0.0,
                "top_terms": []
            }

        # Predict
        prediction = self.classifier.predict([tfidf_vector])[0]
        probabilities = self.classifier.predict_proba([tfidf_vector])[0]
        
        # Get index of prediction to find confidence
        pred_idx = list(self.classifier.classes).index(prediction)
        confidence = probabilities[pred_idx]

        # 2. Get global feature importance
        global_importance = self.classifier.get_feature_importance()

        # 3 & 4. Calculate local contribution and use Min-Heap for Top K
        # We use PriorityQueue(capacity=top_k) for O(U log K) extraction
        pq = PriorityQueue(capacity=top_k)

        for idx, tfidf_score in tfidf_vector.items():
            # Local contribution = TF-IDF * Global Importance of that feature
            contribution = tfidf_score * global_importance[idx]
            
            # Reconstruct word from index
            word = self.tokenizer.get_index_word(idx)
            
            if contribution > 0:
                # Push to min-heap. If it's larger than the smallest in the top K,
                # the heap handles replacement automatically.
                pq.push(contribution, word)

        # 5. Extract top K from heap
        top_terms = []
        for score, word in pq.get_top_k():
            top_terms.append({
                "term": word,
                "contribution_score": score,
                "category_distribution": self.tokenizer.trie.get_category_distribution(word)
            })

        return {
            "prediction": prediction,
            "confidence": confidence,
            "top_terms": top_terms
        }

    def generate_explanation_report(self, text, top_k=5):
        """
        Generate a human-readable text report of the explanation.
        """
        explanation = self.explain_prediction(text, top_k)
        
        report = "=" * 50 + "\n"
        report += "CLASSIFICATION EXPLANATION REPORT\n"
        report += "=" * 50 + "\n"
        
        report += f"Predicted Category : {explanation['prediction']}\n"
        report += f"Confidence Score   : {explanation['confidence'] * 100:.2f}%\n"
        report += "-" * 50 + "\n"
        
        report += f"Top {top_k} Contributing Medical Terms:\n"
        
        if not explanation['top_terms']:
            report += "No vocabulary terms found in document.\n"
        else:
            for i, term_info in enumerate(explanation['top_terms']):
                term = term_info['term']
                score = term_info['contribution_score']
                dist = term_info['category_distribution']
                
                # Format distribution string
                dist_str = ", ".join([f"{k}:{v}" for k, v in dist.items() if v > 0])
                
                report += f"{i+1}. '{term}' (Impact: {score:.4f})\n"
                report += f"    Corpus frequency in categories: {dist_str}\n"
                
        report += "=" * 50 + "\n"
        return report


### 5. Pipeline Integration
DataLoader and the Master Classification Pipeline.

#### 5.1 DataLoader

In [ ]:
"""
Data Loader for Medical Text Classification
===========================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

Handles reading the dataset from CSV, shuffling, and splitting into
training and testing sets. Avoids using pandas for the core logic to
demonstrate fundamental algorithm implementation (like Fisher-Yates shuffle),
though it uses the standard csv module for parsing.
"""

import csv
import random
from collections import Counter


class DataLoader:
    """
    Loads, processes, and splits the medical text dataset.
    """

    def __init__(self, file_path):
        """
        Initialise DataLoader with file path.

        Args:
            file_path (str): Path to the dataset CSV file.
        """
        self.file_path = file_path
        self.data = []    # List of (text, label) tuples
        self.texts = []   # Parallel list of texts
        self.labels = []  # Parallel list of labels

    def load_csv(self, text_col='text', label_col='label'):
        """
        Read the CSV file into memory.

        Args:
            text_col (str): Column name containing the text.
            label_col (str): Column name containing the category label.

        Time Complexity: O(N) where N is number of rows.
        Space Complexity: O(N) memory to store texts and labels.
        """
        self.data = []
        self.texts = []
        self.labels = []

        try:
            with open(self.file_path, mode='r', encoding='utf-8') as file:
                reader = csv.DictReader(file)
                
                # Check if expected columns exist
                if not reader.fieldnames:
                    raise ValueError("CSV file is empty or missing headers.")
                
                # Handle possible alternative column names (Kaggle datasets vary)
                actual_text_col = text_col if text_col in reader.fieldnames else reader.fieldnames[0]
                actual_label_col = label_col if label_col in reader.fieldnames else reader.fieldnames[1]

                for row in reader:
                    text = row.get(actual_text_col, "").strip()
                    label = row.get(actual_label_col, "").strip()
                    
                    if text and label:
                        self.data.append((text, label))
                        self.texts.append(text)
                        self.labels.append(label)
                        
            print(f"Successfully loaded {len(self.data)} records from {self.file_path}")
            return True
            
        except FileNotFoundError:
            print(f"Error: Dataset file not found at {self.file_path}")
            return False
        except Exception as e:
            print(f"Error loading dataset: {e}")
            return False

    def _fisher_yates_shuffle(self, arr):
        """
        In-place Fisher-Yates shuffle algorithm.

        Time Complexity: O(N)
        Space Complexity: O(1)
        """
        n = len(arr)
        for i in range(n - 1, 0, -1):
            j = random.randint(0, i)
            arr[i], arr[j] = arr[j], arr[i]

    def split_data(self, test_ratio=0.2, random_seed=42):
        """
        Shuffle and split data into training and testing sets.

        Args:
            test_ratio (float): Proportion of data to use for testing.
            random_seed (int): Seed for reproducibility.

        Time Complexity: O(N)
        Space Complexity: O(N) for creating split lists.

        Returns:
            tuple: (X_train, y_train, X_test, y_test)
        """
        if not self.data:
            raise ValueError("No data loaded. Call load_csv() first.")

        # Set seed for reproducibility
        random.seed(random_seed)

        # Create indices and shuffle them using our implementation
        indices = list(range(len(self.data)))
        self._fisher_yates_shuffle(indices)

        # Calculate split index
        split_idx = int(len(self.data) * (1 - test_ratio))

        # Split indices
        train_indices = indices[:split_idx]
        test_indices = indices[split_idx:]

        # Create output arrays
        X_train = [self.texts[i] for i in train_indices]
        y_train = [self.labels[i] for i in train_indices]
        X_test = [self.texts[i] for i in test_indices]
        y_test = [self.labels[i] for i in test_indices]

        return X_train, y_train, X_test, y_test

    def get_class_distribution(self):
        """
        Calculate the distribution of classes in the dataset.

        Time Complexity: O(N)
        
        Returns:
            dict: Mapping of class label to count.
        """
        return dict(Counter(self.labels))


#### 5.2 Classification Pipeline

In [ ]:
"""
Master Pipeline for Medical Text Classification
===============================================
Module: COM713 - Advanced Data Structures and Algorithms
Author: [Mohammed Azardeen]
Date: 2026

This module orchestrates the entire classification process, linking the
data structures (Trie, Heap, BST) with the NLP components (Tokenizer,
TF-IDF) and the Ensemble Classifier.

It implements the Queue data structure (using collections.deque) to
process incoming documents in a FIFO manner during inference.
"""











class ClassificationPipeline:
    """
    Master orchestrator for the Medical Text Classification system.
    """

    def __init__(self, data_path, max_vocab_size=5000, min_freq=3):
        """
        Initialise the full pipeline.

        Args:
            data_path (str): Path to the dataset CSV.
            max_vocab_size (int): Max terms in the Trie.
            min_freq (int): Minimum document frequency for a term.
        """
        self.data_path = data_path
        
        # Instantiate components
        self.data_loader = DataLoader(data_path)
        self.preprocessor = TextPreprocessor()
        self.tokenizer = MedicalTokenizer(self.preprocessor, min_freq, max_vocab_size)
        
        # Vectorizer, classifier, evaluator and explainer will be instantiated later
        self.vectorizer = None
        
        # Classifier, evaluator and explainer will be instantiated after vocab is built
        self.classifier = None
        self.evaluator = None
        self.explainer = None
        
        # Queue for batch processing (FIFO)
        self.inference_queue = deque()
        
        self.is_trained = False

    def train(self, test_ratio=0.2):
        """
        Execute the full training pipeline.
        """
        start_time = time.time()
        print("="*50)
        print("STARTING TRAINING PIPELINE")
        print("="*50)

        # 1. Load Data
        print("\n1. Loading Data...")
        if not self.data_loader.load_csv():
            return False
            
        # 2. Split Data
        print("\n2. Splitting Data (Fisher-Yates Shuffle)...")
        X_train, y_train, X_test, y_test = self.data_loader.split_data(test_ratio)
        print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
        
        # 3. Build Vocabulary (Trie)
        print("\n3. Building Medical Vocabulary (Trie)...")
        t_start = time.time()
        self.tokenizer.fit(X_train, y_train)
        print(f"Vocabulary built: {self.tokenizer.get_vocab_size()} unique terms.")
        print(f"Time taken: {time.time() - t_start:.2f}s")
        
        # 4. Feature Engineering (TF-IDF)
        print("\n4. Feature Engineering (TF-IDF Sparse Vectors)...")
        t_start = time.time()
        self.vectorizer = TFIDFVectorizer(self.tokenizer)
        X_train_tfidf = self.vectorizer.fit_transform(X_train)
        print(f"TF-IDF matrices computed.")
        print(f"Time taken: {time.time() - t_start:.2f}s")
        
        # 5. Train Classifier
        print("\n5. Training Ensemble Classifier...")
        t_start = time.time()
        self.classifier = EnsembleClassifier(vocab_size=self.tokenizer.get_vocab_size())
        self.classifier.train(X_train_tfidf, y_train)
        print(f"Time taken: {time.time() - t_start:.2f}s")
        
        # 6. Evaluate Model
        print("\n6. Evaluating on Test Set...")
        X_test_tfidf = [self.vectorizer.transform(doc) for doc in X_test]
        y_pred = self.classifier.predict(X_test_tfidf)
        
        self.evaluator = ModelEvaluator(self.classifier.classes)
        report = self.evaluator.classification_report(y_test, y_pred)
        print("\n" + report)
        
        # 7. Setup Explainer
        self.explainer = ClassificationExplainer(self.tokenizer, self.vectorizer, self.classifier)
        
        self.is_trained = True
        print("="*50)
        print(f"PIPELINE COMPLETE (Total Time: {time.time() - start_time:.2f}s)")
        print("="*50)
        
        return True

    def enqueue_document(self, text):
        """
        Add a document to the inference queue.
        Demonstrates use of the Queue data structure.
        """
        self.inference_queue.append(text)
        print(f"Document added to queue. Queue size: {len(self.inference_queue)}")

    def process_queue(self):
        """
        Process all documents in the inference queue (FIFO).
        """
        if not self.is_trained:
            print("Pipeline must be trained first.")
            return []
            
        results = []
        while self.inference_queue:
            # FIFO: pop from left
            doc = self.inference_queue.popleft()
            
            # Extract features
            tfidf_vec = self.vectorizer.transform(doc)
            
            # Predict
            pred = self.classifier.predict([tfidf_vec])[0]
            results.append((doc, pred))
            
        return results

    def explain(self, text, top_k=5):
        """
        Explain a single prediction.
        """
        if not self.is_trained:
            return "Pipeline must be trained first."
            
        return self.explainer.generate_explanation_report(text, top_k)


### 6. Execution and Evaluation
Running the full pipeline.

In [13]:
# Initialize the pipeline
pipeline = ClassificationPipeline(data_path='dataset/refined_medical_dataset.csv', max_vocab_size=5000, min_freq=2)

# IMPORTANT: You must place the dataset in the 'dataset' folder.
# Download from Kaggle: https://www.kaggle.com/datasets/praveengovi/medical-text-classification

# Uncomment to run training (if dataset is available)
pipeline.train(test_ratio=0.2)


STARTING TRAINING PIPELINE

1. Loading Data...
Successfully loaded 3000 records from dataset/refined_medical_dataset.csv

2. Splitting Data (Fisher-Yates Shuffle)...
Train size: 2400 | Test size: 600

3. Building Medical Vocabulary (Trie)...
Vocabulary built: 44 unique terms.
Time taken: 0.03s

4. Feature Engineering (TF-IDF Sparse Vectors)...
TF-IDF matrices computed.
Time taken: 0.06s

5. Training Ensemble Classifier...
Converting 2400 sparse vectors to dense matrix...
Training Random Forest ensemble...
Training complete.
Time taken: 0.07s

6. Evaluating on Test Set...

Class                          | Precision  | Recall     | F1-Score  
--------------------------------------------------------------------
diagnosis                      | 1.0000     | 1.0000     | 1.0000
patient history                | 1.0000     | 1.0000     | 1.0000
treatment                      | 1.0000     | 1.0000     | 1.0000
--------------------------------------------------------------------
Macro Avg      

True

### 7. Explainability Demonstration

In [14]:
# Explain a prediction
sample_text = "The patient presented with severe chest pain, elevated heart rate, and hypertension. ECG showed abnormalities."

# Uncomment to run (if pipeline is trained)
print(pipeline.explain(sample_text, top_k=5))


CLASSIFICATION EXPLANATION REPORT
Predicted Category : diagnosis
Confidence Score   : 77.00%
--------------------------------------------------
Top 5 Contributing Medical Terms:
1. 'patient' (Impact: 0.0340)
    Corpus frequency in categories: patient history:810, diagnosis:802
2. 'pain' (Impact: 0.0011)
    Corpus frequency in categories: diagnosis:221
3. 'hypertension' (Impact: 0.0003)
    Corpus frequency in categories: patient history:212, diagnosis:151

